In [1]:
import sys
import numpy as np
from scipy.optimize import linprog
import itertools
import os
import time

In [2]:
import random
import math

In [3]:
random.seed(42)

In [4]:
tests = ['fl_25_2', 'fl_100_1', 'fl_200_7', 'fl_500_7', 'fl_1000_2', 'fl_2000_2']
thresholds = [(4000000, 3269822), (26000000, 22724634), (5000000, 4711295), (30000000, 27006099), (10000000, 8879294), (10000000, 7453531)]

In [16]:
def load_data(test):
    with open(f"data/{test}") as file:
        lines = file.readlines()
        n, m = list(map(int, lines[0].split()))
        price = list()
        cap = list()
        shops = list()
        for i in range(n):
            p, c, x, y = list(map(float, lines[1 + i].split()))
            price.append(p)
            cap.append(c)
            shops.append((x, y))
            
        calls = list()
        customers = list()
        for i in range(m):
            call, x, y = list(map(float, lines[1 + n + i].split()))
            calls.append(call)
            customers.append((x, y))
            
        return n, m, price, cap, shops, calls, customers

Я не понял, почему здесь нужно как - то формулировать задачу, ибо вроде здесь и так понятна формальная постановка:

Нужно выбрать множество магазинов $S$ и $F: [M] \rightarrow S$, которая будет задавать для каждого покупателя магазин, в который он будет ходить, причем минимизация будет происходить по $$\sum_{i \in S} s_i + \sum_{i=1}^{M} \text{dist}(\text{customer}_i, \text{shop}_{F_i})$$

Требуется выполнить условия:
$$\forall i \in [N]: \sum_{x \in F^{-1}(i)} d_x \leq cap_i$$

In [17]:
def check_facility(n, m, price, cap, shops, calls, customers, choise, f):
    if len(set(choise)) != len(choise):
        raise Exception("Not correct facility")

    if len(choise) == 0:
        raise Exception("Not correct facility")

    if min(choise) < 0 or max(choise) >= n:
        raise Exception("Not correct facility")

    result = 0
    for i in choise:
        result += price[i]

    sum_cap = [0] * n
    for i in range(m):
        shop = choise[f[i]]
        sum_cap[shop] += calls[i]
        
        result += math.dist(customers[i], shops[shop])

    for i in range(n):
        if sum_cap[i] > cap[i]:
            raise Exception("Not correct facility")

    return result

In [18]:
def passed_cnt(result, idx):
    if result <= thresholds[idx][1]:
        return 2
    elif result <= thresholds[idx][0]:
        return 1
    else:
        return 0

In [19]:
def test_method(method, name, use_file=False):
    print(f"Checking {name}")
    score = 0
    for i, test in enumerate(tests):
        n, m, price, cap, shops, calls, customers = load_data(test)
        start = time.time()
        
        if not use_file:
            choise, f = method(n, m, price, cap, shops, calls, customers)
        else:
            choise, f = method(test)

        end = time.time()
        elapsed = end - start
        print(f"Execution time: {elapsed:.4f} seconds")
            
        result = check_facility(n, m, price, cap, shops, calls, customers, choise, f)
        passed = passed_cnt(result, i)
        
        if passed == 1:
            score += 3
        elif passed == 2:
            score += 5

        print(f"Target function {test}: {result}")
        print(f"Passed {test}: {passed}")

    print(f"Score: {score}")

Идея:
\
Пусть множество магазинов фиксированно. Подберем выбор покупателям из следующих жадных соображений:
в порядке убывания их $d_c$ будем назначать их в магазин с минимальным расстоянием, чтобы это не превышало $\text{cap}$ магазина
\
Множество магазинов выбирать будем таким образом: берем случайный порядок магазинов и добавляем магазин, пока не получится распределить всех покупателей. В качестве инициализации возьмем порядок по возрастанию стоимости открытия.
\
Дальше будем работать с лучшим решением. Удаляем случайное подмножество магазинов и добавляем в случайном порядке магазины, пока решение не станет валидным.

In [88]:
!g++ -std=c++2a cpp_methods/greedy.cpp -o tmp/greedy

In [89]:
def greedy_facility(test_file): 
    os.system(f"./tmp/greedy data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        choice = list(map(int, lines[0].split()))
        f = list(map(int, lines[1].split()))
        return choice, f

In [90]:
test_method(greedy_facility, "greedy", True)

Checking greedy
Execution time: 120.2355 seconds
Target function fl_25_2: 3269821.3205308816
Passed fl_25_2: 2
Execution time: 120.0232 seconds
Target function fl_100_1: 24011515.28226571
Passed fl_100_1: 1
Execution time: 120.1057 seconds
Target function fl_200_7: 5166400.781836117
Passed fl_200_7: 0
Execution time: 121.7602 seconds
Target function fl_500_7: 33782244.05668253
Passed fl_500_7: 0
Execution time: 121.0050 seconds
Target function fl_1000_2: 11584460.191694617
Passed fl_1000_2: 0
Execution time: 148.6925 seconds
Target function fl_2000_2: 10640339.788533332
Passed fl_2000_2: 0
Score: 8


Пока останемся на идее, что для фиксированного множества магазинов задача будет решаться жадно и нужно подобрать хороший набор магазинов.
Мы умеем понятным образом сравнивать множества допустимых магазинов, но недопустимые множества мы пока никак не различаем. Предлагается ввести штраф за каждого необработанного клиента $\lambda$. Теперь все множества сравнимы численно. 
Сделаем отжиг на множество магазинов, где изменением будет 1-flip (добавление/удаление 1 магазина).

In [183]:
!g++ -std=c++2a cpp_methods/annealing.cpp -o tmp/annealing

In [184]:
def sa_facility(test_file): 
    os.system(f"./tmp/annealing data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        choice = list(map(int, lines[0].split()))
        f = list(map(int, lines[1].split()))
        return choice, f

In [185]:
test_method(sa_facility, "sa", True)

Checking sa
Execution time: 60.4558 seconds
Target function fl_25_2: 3269821.3205308816
Passed fl_25_2: 2
Execution time: 60.7185 seconds
Target function fl_100_1: 23934694.668194585
Passed fl_100_1: 1
Execution time: 60.0569 seconds
Target function fl_200_7: 4869605.026917702
Passed fl_200_7: 1
Execution time: 95.4206 seconds
Target function fl_500_7: 28767771.868400544
Passed fl_500_7: 1
Execution time: 85.9542 seconds
Target function fl_1000_2: 9452702.496348307
Passed fl_1000_2: 1
Execution time: 78.5240 seconds
Target function fl_2000_2: 8149920.570414765
Passed fl_2000_2: 1
Score: 20


Теперь прошли все простые пороги и один сложный